In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

data = pd.read_csv("customer_churn_large.csv")
data = pd.get_dummies(
    data,
    columns=["contract_type", "internet_service"],
    drop_first=True
)

X = data.drop("churn", axis=1)
y = data["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))
print(classification_report(y_test, lr_pred))

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))
print(classification_report(y_test, rf_pred))

importance = pd.Series(rf_model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n===== Top 10 Important Features =====")
print(importance.head(10))

data_test = X_test.copy()
data_test["Churn_Probability"] = rf_prob

high_risk_customers = data_test[data_test["Churn_Probability"] > 0.7]

print("\n===== High Risk Customers Sample =====")
print(high_risk_customers.head())


===== Logistic Regression =====
Accuracy: 0.7
ROC-AUC: 0.5166666666666666
              precision    recall  f1-score   support

          No       0.70      1.00      0.82        70
         Yes       0.00      0.00      0.00        30

    accuracy                           0.70       100
   macro avg       0.35      0.50      0.41       100
weighted avg       0.49      0.70      0.58       100



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



===== Random Forest =====
Accuracy: 0.71
ROC-AUC: 0.5328571428571429
              precision    recall  f1-score   support

          No       0.73      0.94      0.82        70
         Yes       0.56      0.17      0.26        30

    accuracy                           0.71       100
   macro avg       0.64      0.55      0.54       100
weighted avg       0.67      0.71      0.65       100


===== Top 10 Important Features =====
total_charges             0.218199
customer_id               0.209243
monthly_charges           0.197563
tenure                    0.195612
support_calls             0.093363
contract_type_Two Year    0.029159
internet_service_Fiber    0.028693
contract_type_One Year    0.028169
dtype: float64

===== High Risk Customers Sample =====
Empty DataFrame
Columns: [customer_id, tenure, monthly_charges, total_charges, support_calls, contract_type_One Year, contract_type_Two Year, internet_service_Fiber, Churn_Probability]
Index: []
